# 函数式编程
- 面向过程的程序设计
- 高抽象程度编程范式
- 可以将函数本身作为参数传入另外一个函数，还允许返回另外一个函数

## 高阶函数
- 变量可以指向函数
- 函数名也是变量(一旦对函数名赋值了，想要恢复函数，需要重启python环境或者执行del（del str）)
- 一个函数可以作为另外一个函数的参数

In [13]:
from random import random
from typing import Iterable

print(max)
f0 = max
print(f0)

abs=1
print(abs)

print()
f1 = max
f2 = min
def func(x, y, f):
    print(f(x, y))
func(1, 2, f1)
func(1, 2, f2)


<built-in function max>
<built-in function max>
1

2
1


### map/reduce
- map的两个参数——函数， Iterable——返回Iterable
- reduce(f, [x1, x2, x3, x4]) = f(f(f(x1, x2), x3), x4)——返回Iterable

In [47]:
from collections.abc import Iterable
from functools import reduce


print('-'*10, "map——将传入的函数作用到序列的每一个元素——返回一个Iterable",'-'*10)
def f(x):
    return x * x
my_list = list(range(10))
print(my_list)
res_list = map(f, my_list)

print(list(res_list))
print(list(map(str, my_list)))
print(isinstance(map(f, my_list), Iterable))

# reduce(f, [x1, x2, x3, x4]) = f(f(f(x1, x2), x3), x4)
print('-'*10, "reduce——把结果继续和下一个元素做累计计算——返回一个单一值",'-'*12)

def add(x, y):
    return x + y
add_list = list(range(10))
print(reduce(add, add_list))

print('-'*30, "组合用法",'-'*30)
my_str = "123456"
def str2int(input_str):
    def fn(x,y):
        return x * 10 + y
    def char2num(ch):
        digits = {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}
        return digits[ch]
    return reduce(fn, list(map(char2num, input_str)))
print(str2int(my_str))

---------- map——将传入的函数作用到序列的每一个元素——返回一个Iterable ----------
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
True
---------- reduce——把结果继续和下一个元素做累计计算——返回一个单一值 ------------
45
------------------------------ 组合用法 ------------------------------
123456


### filter
- 用于过滤一个序列
- 接受一个函数和Iterable，返回一个Iterable

In [48]:
# 生成奇数
def _odd_iter():
    n = 1
    while True:
        n = n + 2
        yield n
# 筛选出被整除的数，留下不被整除的数
def _not_divisible(n):
    return lambda x : x % n > 0
def primes():
    yield 2# 第一个素数
    it = _odd_iter()# 后续的素数必然都是奇数
    while True:
        n = next(it)
        yield n
        # 构建出新数列，这里每一次调用next()，都会套上一层filter
        # it = filter(不能被 3 整除, _odd_iter())
        # it = filter(不能被 5 整除, filter(不能被 3 整除, _odd_iter()))
        # it = filter(不能被 7 整除, filter(不能被 5 整除, filter(不能被 3 整除, _odd_iter())))
        it = filter(_not_divisible(n), it)

# 打印100以内的素数:
for n in primes():
    if n < 100:
        print(n)
    else:
        break


2
3
5
7
11
13
17
19
23
29
31
37
41
43
47
53
59
61
67
71
73
79
83
89
97


### sorted
- 内置 sorted()——可对list排序，默认升序——返回一个新list，而不是作用于原数据上（区别于C++）
- key参数——自定义排序——key指定的函数将作用于list的每一个元素上
- reverse 逆序

In [55]:
my_list = [1,20,31,-23,-6,9,0]
print(sorted(my_list))
print(my_list)

def my_abs(x):
    if x >= 0:
        return x
    else:
        return -x
print(sorted(my_list, key=my_abs))
print(sorted(my_list, key=my_abs,reverse=True))

[-23, -6, 0, 1, 9, 20, 31]
[1, 20, 31, -23, -6, 9, 0]
[0, 1, -6, 9, 20, -23, 31]
[31, -23, 20, 9, -6, 1, 0]


## 返回函数
- 函数可以作为结果值返回——不需要立刻得到结果

In [62]:
def lazy_sum(*args):
    def my_sum():
        ax = 0
        for n in args:
            ax = ax + n
        return ax
    return my_sum

my_list = list(range(10))
f = lazy_sum(*my_list)
print(f)
print(f())

<function lazy_sum.<locals>.my_sum at 0x00000230E2F1FBA0>
45
9 9 9
1 4 9


- ***闭包***
> 内部函数可以引用外部函数，当返回时，相关参数和变量都保存在返回的函数中——“闭包”
>
> 返回闭包时牢记一点：返回函数不要引用任何循环变量，或者后续会发生变化的变量。

In [68]:
# 返回的函数引用了变量i，但它并非立刻执行。
# 等到3个函数都返回时，它们所引用的变量i已经变成了3，因此最终结果为9
def count():
    fs = []
    for i in range(1, 4):
        def f():
             return i*i
        fs.append(f)
    return fs

my_fn = count()
print(my_fn[0](), my_fn[1](), my_fn[2]())

# 在循环中执行 f(i) 时，当前循环的 i 的值被立即传递给了形参 j。
# 最内层的 g() 闭包捕获的是自己外层那个特定上下文里的 j。
# 此时 j 分别被永久冻结在了 1, 2, 3
def count():
    def f(j):
        def g():
            return j*j
        return g
    fs = []
    for i in range(1, 4):
        fs.append(f(i)) # f(i)立刻被执行，因此i的当前值被传入f()
    return fs
my_fn = count()
print(my_fn[0](), my_fn[1](), my_fn[2]())

9 9 9
1 4 9


- ***nonlocal***
> 在内部函数fn()的视角，x作为局部变量并没有初始化，直接计算x+1是不行的
>
> 加上nonlocal x这个声明后，解释器把fn()的x看作外层函数的局部变量，它已经被初始化了

In [63]:
def inc():
    x = 0
    def fn():
        # 仅读取x的值:
        return x + 1
    return fn

f = inc()
print(f()) # 1
print(f()) # 1

def inc():
    x = 0
    def fn():
        nonlocal x
        x = x + 1
        return x
    return fn

f = inc()
print(f()) # 1
print(f()) # 2


1
1
1
2


## 匿名函数
> 关键字lambda表示匿名函数，冒号前面的参数表示函数参数

In [72]:
f = lambda x : x*x
print(f)
print(f(5))

# 把匿名函数作为返回值返回
def build(x, y):
    return lambda: x * x + y * y


def is_odd(n):
    return n % 2 == 1

L = list(filter(lambda n : n % 2 == 1, range(1, 20)))

print(L)

<function <lambda> at 0x00000230E2F1E3E0>
25
[1, 3, 5, 7, 9, 11, 13, 15, 17, 19]


## 装饰器
> 在代码运行期间动态增加功能的方式，称之为“装饰器”（Decorator）
>
> 本质上，decorator就是一个返回函数的高阶函数
>
> 把@log放到now()函数的定义处，相当于执行了语句:now = log(now)

In [75]:
def log(func):
    def wrapper(*args, **kw):
        print('call %s():' % func.__name__)
        return func(*args, **kw)
    return wrapper

# 把@log放到now()函数的定义处，相当于执行了语句：now = log(now)
# 同名的now变量指向了新的函数
@log
def now():
    print('2024-6-1')
now()

call now():
2024-6-1


- 如果decorator本身需要传入参数，那就需要编写一个返回decorator的高阶函数
- 在定义wrapper()的前面加上@functools.wraps(func)
- 因为返回的那个wrapper()函数名字就是'wrapper'，
- 所以，需要把原始函数的__name__等属性复制到wrapper()函数中，否则，有些依赖函数签名的代码执行就会出错。

In [83]:
def log(text):
    def decorator(func):
        def wrapper(*args, **kw):
            print('%s %s():' % (text, func.__name__))
            return func(*args, **kw)
        return wrapper
    return decorator
# 等价于now = log('execute')(now)
@log('execute')
def now():
    print('2024-6-1')
now()
print(now.__name__)

print('-'*50)
import functools

def log(text):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kw):
            print('%s %s():' % (text, func.__name__))
            return func(*args, **kw)
        return wrapper
    return decorator
# 等价于now = log('execute')(now)
@log('execute')
def now():
    print('2024-6-1')
now()
print(now.__name__)

execute now():
2024-6-1
wrapper
--------------------------------------------------
execute now():
2024-6-1
now


## 偏函数
- functools.partial的作用就是，把一个函数的某些参数给固定住（也就是设置默认值），返回一个新的函数
- 在调用时，这些被固定的值也还是可以修改的

In [85]:
import functools
int2 = functools.partial(int, base=2)
print(int2('1000000'))
print(int2('1010101'))

print(int2('1000000', base=10))

64
85
1000000
